In [ ]:
# ============================================================
# CELL 1: Environment Setup + Install Dependencies
# ============================================================

import os
import sys
import subprocess

print("=" * 60)
print("STAGE 1/5: ENVIRONMENT SETUP")
print("=" * 60)

# Verify GPU
import torch
print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU not available! Set Accelerator to GPU T4 x2")

# Verify dataset
DFIRE_PATH = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data'
if not os.path.exists(DFIRE_PATH):
    raise RuntimeError(f"Dataset missing at {DFIRE_PATH}")

splits = {}
for split in ['train', 'val', 'test']:
    img_dir = f'{DFIRE_PATH}/{split}/images'
    splits[split] = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0

print(f"\nDataset:")
print(f"  Train: {splits['train']} images")
print(f"  Val:   {splits['val']} images")
print(f"  Test:  {splits['test']} images")

assert splits['train'] == 14122, "Train count mismatch"
assert splits['val'] == 3099, "Val count mismatch"
assert splits['test'] == 4306, "Test count mismatch"

# Install YOLOX dependencies (idempotent - safe to re-run)
print("\nInstalling dependencies...")
packages = ['loguru', 'pycocotools', 'tabulate', 'thop', 'tqdm', 'ninja', 'opencv-python']
for pkg in packages:
    result = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True, text=True)
    print(f"  {'✓' if result.returncode == 0 else '✗'} {pkg}")

print("\n✓ STAGE 1 COMPLETE")

In [ ]:
# ============================================================
# CELL 2: Clone YOLOX + Configure Python Paths
# ============================================================

import os
import sys
import shutil

print("=" * 60)
print("STAGE 2/5: YOLOX SETUP")
print("=" * 60)

YOLOX_DIR = '/kaggle/working/YOLOX'

# Clean clone (in case any previous state exists)
if os.path.exists(YOLOX_DIR):
    shutil.rmtree(YOLOX_DIR)
    print(f"Removed existing {YOLOX_DIR}")

# Clone fresh
os.chdir('/kaggle/working')
result = os.system('git clone https://github.com/Megvii-BaseDetection/YOLOX.git --quiet')
if result != 0:
    raise RuntimeError("Failed to clone YOLOX")
print(f"✓ Cloned YOLOX to {YOLOX_DIR}")

# Add to Python path
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
print(f"✓ Added {YOLOX_DIR} to sys.path")

# Set PYTHONPATH for subprocesses (training script needs this)
os.environ['PYTHONPATH'] = f'{YOLOX_DIR}:{os.environ.get("PYTHONPATH", "")}'
print(f"✓ Set PYTHONPATH for subprocesses")

# Clear any cached yolox modules
for m in list(sys.modules.keys()):
    if m.startswith('yolox'):
        del sys.modules[m]
print(f"✓ Cleared yolox module cache")

print("\n✓ STAGE 2 COMPLETE")

In [ ]:
# ============================================================
# CELL 2.5: Clean up previous outputs
# ============================================================

import os
import shutil
import sys

print("=" * 60)
print("CLEANING UP PREVIOUS OUTPUTS")
print("=" * 60)

# Remove old training outputs
output_dir = '/kaggle/working/YOLOX_outputs'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print(f"✓ Removed: {output_dir}")
else:
    print(f"  No existing outputs to clean")

# Remove old final_models
final_dir = '/kaggle/working/final_models'
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)
    print(f"✓ Removed: {final_dir}")

# Clear yolox module cache
for m in list(sys.modules.keys()):
    if m.startswith('yolox'):
        del sys.modules[m]
print("✓ Cleared yolox module cache")

print("\n✓ CLEANUP COMPLETE - Ready for fresh training")

In [ ]:
# ============================================================
# CELL 3: Create LCAM Module + Patch YOLOX Backbone
# DPU-Optimized v3.2 - Fixed compilation hang
# ============================================================

import os

YOLOX_DIR = '/kaggle/working/YOLOX'

print("=" * 60)
print("STAGE 3/5: LCAM INTEGRATION (DPU-OPTIMIZED v3.2)")
print("=" * 60)

lcam_code = '''"""
LCAM v3.2 - DPU-Optimized Lightweight Channel + Spatial Attention

Changes from v3.1:
- Replaced torch.max(dim=1) in SpatialAttention with Conv2d
- torch.max over 512 channels caused vai_c_xir to hang 40+ min
- Both channel reductions now use Conv2d (conv2d-fix.so)
- 100% DPU-compilable, no compilation hang
"""

import torch
import torch.nn as nn


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, reduced, 1, bias=False),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(reduced, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        # torch.max over H then W (NOT over channels)
        # Spatial dims are small (max 20x20), compiler handles easily
        max_h = torch.max(x, dim=2, keepdim=True)[0]
        max_pool = torch.max(max_h, dim=3, keepdim=True)[0]
        max_out = self.mlp(max_pool)
        return x * self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        padding = (kernel_size - 1) // 2

        # Both reductions use Conv2d (DPU-friendly)
        self.channel_reduce_avg = nn.Conv2d(channels, 1, 1, bias=False)
        self.channel_reduce_max = nn.Conv2d(channels, 1, 1, bias=False)

        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.channel_reduce_avg(x)
        max_out = self.channel_reduce_max(x)  # No torch.max over channels!
        spatial_input = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(spatial_input))


class LCAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(channels, spatial_kernel)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x
'''

with open(f'{YOLOX_DIR}/yolox/models/lcam.py', 'w') as f:
    f.write(lcam_code)
print("✓ LCAM v3.2 created")

# ─── Patch darknet.py ───
darknet_path = f'{YOLOX_DIR}/yolox/models/darknet.py'
with open(darknet_path, 'r') as f:
    text = f.read()

if 'from .lcam import LCAM' not in text:
    text = text.replace(
        'from .network_blocks import',
        'from .lcam import LCAM\nfrom .network_blocks import',
        1
    )

text = text.replace(
    'self.stem = Focus(3, base_channels, ksize=3, act=act)',
    'self.stem = BaseConv(3, base_channels, 3, 2, act=act)'
)

csp_idx = text.find('class CSPDarknet')
before_csp = text[:csp_idx]
csp_block = text[csp_idx:]

if 'self.lcam2 = LCAM' not in csp_block:
    fwd_idx = csp_block.find('    def forward(self, x)')
    lcam_init = '''
        # LCAM v3.2 modules
        self.lcam2 = LCAM(base_channels * 2)
        self.lcam3 = LCAM(base_channels * 4)
        self.lcam4 = LCAM(base_channels * 8)
        self.lcam5 = LCAM(base_channels * 16)

'''
    csp_block = csp_block[:fwd_idx] + lcam_init + csp_block[fwd_idx:]

    replacements = [
        ('        x = self.dark2(x)\n        outputs["dark2"] = x',
         '        x = self.dark2(x)\n        x = self.lcam2(x)\n        outputs["dark2"] = x'),
        ('        x = self.dark3(x)\n        outputs["dark3"] = x',
         '        x = self.dark3(x)\n        x = self.lcam3(x)\n        outputs["dark3"] = x'),
        ('        x = self.dark4(x)\n        outputs["dark4"] = x',
         '        x = self.dark4(x)\n        x = self.lcam4(x)\n        outputs["dark4"] = x'),
        ('        x = self.dark5(x)\n        outputs["dark5"] = x',
         '        x = self.dark5(x)\n        x = self.lcam5(x)\n        outputs["dark5"] = x'),
    ]
    for old, new in replacements:
        csp_block = csp_block.replace(old, new)

text = before_csp + csp_block

with open(darknet_path, 'w') as f:
    f.write(text)
print("✓ darknet.py patched")

# ─── Set LeakyReLU ───
base_path = f'{YOLOX_DIR}/yolox/exp/yolox_base.py'
with open(base_path, 'r') as f:
    content = f.read()
content = content.replace('self.act = "silu"', 'self.act = "lrelu"')
with open(base_path, 'w') as f:
    f.write(content)
print("✓ LeakyReLU set")

# ─── Disable mlflow ───
mlflow_path = f'{YOLOX_DIR}/yolox/utils/mlflow_logger.py'
if os.path.exists(mlflow_path):
    with open(mlflow_path, 'r') as f:
        c = f.read()
    if 'import importlib.metadata' in c and '# import importlib.metadata' not in c:
        c = c.replace('import importlib.metadata', '# import importlib.metadata')
        with open(mlflow_path, 'w') as f:
            f.write(c)
        print("✓ mlflow disabled")

# ─── Wrap thop profiler ───
model_utils_path = f'{YOLOX_DIR}/yolox/utils/model_utils.py'
if os.path.exists(model_utils_path):
    with open(model_utils_path, 'r') as f:
        c = f.read()
    old = 'flops, params = profile(deepcopy(model), inputs=(img,), verbose=False)'
    if old in c and 'except Exception' not in c:
        new = '''try:
        flops, params = profile(deepcopy(model), inputs=(img,), verbose=False)
    except Exception as e:
        flops = 0
        params = sum(p.numel() for p in model.parameters())'''
        c = c.replace(old, new)
        with open(model_utils_path, 'w') as f:
            f.write(c)
        print("✓ thop profiler wrapped")

# ─── Verify model ───
import sys
import torch

for m in list(sys.modules.keys()):
    if m.startswith('yolox'):
        del sys.modules[m]

from yolox.exp import get_exp
exp = get_exp(exp_file=None, exp_name='yolox-s')
exp.num_classes = 2
exp.act = 'lrelu'
model = exp.get_model()
model.eval()

total = sum(p.numel() for p in model.parameters())
lcam_count = sum(1 for _, m in model.named_modules() if m.__class__.__name__ == 'LCAM')

print(f"\nModel verification:")
print(f"  Parameters: {total:,} ({total/1e6:.2f}M)")
print(f"  LCAM count: {lcam_count}/4")
assert lcam_count == 4

# Verify v3.2 - check for channel_reduce_max
has_max_conv = any('channel_reduce_max' in n for n, _ in model.named_parameters())
print(f"  Has channel_reduce_max: {has_max_conv} ← v3.2 check")
assert has_max_conv, "v3.2 channel_reduce_max not found!"

with torch.no_grad():
    out = model(torch.randn(1, 3, 640, 640))
print(f"  Forward 640x640: ✓")

print("\n✓ STAGE 3 COMPLETE (LCAM v3.2)")

In [ ]:
# ============================================================
# CELL 4: Dataset Conversion + Pretrained Weights + Config
# ============================================================

import os
import json
from PIL import Image
from pathlib import Path
from tqdm import tqdm

print("=" * 60)
print("STAGE 4/5: DATASET + WEIGHTS + CONFIG")
print("=" * 60)

DFIRE_PATH = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data'
COCO_DIR = '/kaggle/working/coco_dataset'

os.makedirs(f'{COCO_DIR}/annotations', exist_ok=True)

# ─── YOLO → COCO conversion (skip if already done) ───
def yolo_to_coco(images_dir, labels_dir, output_json, split):
    if os.path.exists(output_json):
        print(f"  - {split} already converted, skipping")
        return
    
    coco = {
        'images': [],
        'annotations': [],
        'categories': [
            {'id': 1, 'name': 'smoke', 'supercategory': 'fire_detection'},
            {'id': 2, 'name': 'fire', 'supercategory': 'fire_detection'},
        ]
    }
    
    ann_id = 1
    image_files = sorted(
        list(Path(images_dir).glob('*.jpg')) + 
        list(Path(images_dir).glob('*.png'))
    )
    
    for img_id, img_path in enumerate(tqdm(image_files, desc=split, leave=False), start=1):
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except:
            continue
        
        coco['images'].append({
            'id': img_id,
            'file_name': img_path.name,
            'width': width, 'height': height
        })
        
        label_path = Path(labels_dir) / (img_path.stem + '.txt')
        if not label_path.exists():
            continue
        
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                try:
                    cls, xc, yc, w, h = int(float(parts[0])), *map(float, parts[1:])
                except ValueError:
                    continue
                
                abs_w = w * width
                abs_h = h * height
                abs_x = (xc * width) - (abs_w / 2)
                abs_y = (yc * height) - (abs_h / 2)
                
                if abs_w <= 0 or abs_h <= 0:
                    continue
                
                coco['annotations'].append({
                    'id': ann_id, 'image_id': img_id,
                    'category_id': cls + 1,
                    'bbox': [abs_x, abs_y, abs_w, abs_h],
                    'area': abs_w * abs_h,
                    'iscrowd': 0, 'segmentation': []
                })
                ann_id += 1
    
    with open(output_json, 'w') as f:
        json.dump(coco, f)
    print(f"  ✓ {split}: {len(coco['images'])} images, {len(coco['annotations'])} annotations")

print("\nConverting YOLO → COCO format...")
yolo_to_coco(f'{DFIRE_PATH}/train/images', f'{DFIRE_PATH}/train/labels',
             f'{COCO_DIR}/annotations/instances_train.json', 'train')
yolo_to_coco(f'{DFIRE_PATH}/val/images', f'{DFIRE_PATH}/val/labels',
             f'{COCO_DIR}/annotations/instances_val.json', 'val')
yolo_to_coco(f'{DFIRE_PATH}/test/images', f'{DFIRE_PATH}/test/labels',
             f'{COCO_DIR}/annotations/instances_test.json', 'test')

# ─── Create image symlinks ───
print("\nCreating image symlinks...")
for split, src_split in [('train2017', 'train'), ('val2017', 'val'), ('test2017', 'test')]:
    dst = f'{COCO_DIR}/{split}'
    os.makedirs(dst, exist_ok=True)
    os.system(f'ln -sf {DFIRE_PATH}/{src_split}/images/* {dst}/ 2>/dev/null')
    print(f"  ✓ {split}: {len(os.listdir(dst))} files linked")

# ─── Download pretrained weights ───
PRETRAINED = '/kaggle/working/pretrained/yolox_s.pth'
os.makedirs(os.path.dirname(PRETRAINED), exist_ok=True)

if not os.path.exists(PRETRAINED):
    print("\nDownloading pretrained YOLOX-S weights...")
    os.system(f'wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth -O {PRETRAINED}')
    print(f"  ✓ Downloaded ({os.path.getsize(PRETRAINED)/1e6:.1f} MB)")
else:
    print(f"\n✓ Pretrained weights already exist ({os.path.getsize(PRETRAINED)/1e6:.1f} MB)")

# ─── Create training config ───
YOLOX_DIR = '/kaggle/working/YOLOX'
config_path = f'{YOLOX_DIR}/exps/custom/lcam_yolox_wildfire.py'
os.makedirs(os.path.dirname(config_path), exist_ok=True)

config = '''import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.50
        self.act = "lrelu"
        self.num_classes = 2
        
        self.data_dir = "/kaggle/working/coco_dataset"
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"
        self.name = "train2017"
        
        self.max_epoch = 100
        self.basic_lr_per_img = 0.01 / 64.0
        self.momentum = 0.937
        self.weight_decay = 0.0005
        self.warmup_epochs = 5
        self.no_aug_epochs = 15
        
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.multiscale_range = 5
        
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        
        self.data_num_workers = 2
        self.eval_interval = 5
        self.print_interval = 50
        
        self.output_dir = "/kaggle/working/YOLOX_outputs"
        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]
'''
with open(config_path, 'w') as f:
    f.write(config)
print(f"\n✓ Training config: {config_path}")

print("\n✓ STAGE 4 COMPLETE")

In [ ]:
# Replace Cell 5 with this:

import os

print("=" * 60)
print("STAGE 5A/5: TRAINING CONFIG (SPEED OPTIMIZED)")
print("=" * 60)

YOLOX_DIR = '/kaggle/working/YOLOX'
config_path = f'{YOLOX_DIR}/exps/custom/lcam_yolox_wildfire.py'

config = '''import os
from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super().__init__()
        
        # MODEL
        self.depth = 0.33
        self.width = 0.50
        self.act = "lrelu"
        self.num_classes = 2
        
        # DATASET
        self.data_dir = "/kaggle/working/coco_dataset"
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"
        self.name = "train2017"
        
        # SHORTER TRAINING (will still get good results)
        self.max_epoch = 30
        self.warmup_epochs = 3
        self.no_aug_epochs = 10
        self.eval_interval = 10
        self.print_interval = 50
        
        self.basic_lr_per_img = 0.01 / 64.0
        self.momentum = 0.937
        self.weight_decay = 0.0005
        
        # FIXED INPUT SIZE (CRITICAL - no multiscale!)
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.multiscale_range = 0
        
        # LIGHTER AUGMENTATION (faster batches)
        self.mosaic_prob = 0.5
        self.mixup_prob = 0.5
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.degrees = 10.0
        self.translate = 0.1
        self.mosaic_scale = (0.5, 1.5)
        self.mixup_scale = (0.5, 1.5)
        self.shear = 2.0
        self.enable_mixup = True
        
        self.test_conf = 0.001
        self.nmsthre = 0.65
        
        # MORE WORKERS
        self.data_num_workers = 4
        
        self.output_dir = "/kaggle/working/YOLOX_outputs"
        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]
'''

with open(config_path, 'w') as f:
    f.write(config)

print("✓ Optimized config saved")
print("\nKey changes:")
print("  multiscale_range: 0 (was 5)")
print("  mosaic_prob: 0.5 (was 1.0)")  
print("  mixup_prob: 0.5 (was 1.0)")
print("  data_num_workers: 4 (was 2)")
print("  max_epoch: 30 (was 50)")
print("\nExpected iter_time: ~0.5-1s")
print("Expected total time: ~2-3 hours")

In [ ]:
import os
import sys
import subprocess
import time

print("=" * 60)
print("STAGE 5B/5: TRAINING (OPTIMIZED)")
print("=" * 60)

YOLOX_DIR = '/kaggle/working/YOLOX'
EXP_FILE = f'{YOLOX_DIR}/exps/custom/lcam_yolox_wildfire.py'
PRETRAINED = '/kaggle/working/pretrained/yolox_s.pth'

os.chdir(YOLOX_DIR)
env = os.environ.copy()
env['PYTHONPATH'] = f'{YOLOX_DIR}:{env.get("PYTHONPATH", "")}'

# CRITICAL: batch_size = 32 (was 16)
cmd = [
    'python3', 'tools/train.py',
    '-f', EXP_FILE,
    '-d', '1',
    '-b', '32',           # ← 2x batch size
    '--fp16',
    '-c', PRETRAINED,
]

print(f"Batch size: 32 (was 16)")
print(f"Starting training...\n")

start = time.time()
process = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True, bufsize=1
)

for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)

elapsed = time.time() - start
print(f"\nFinished in {elapsed/60:.1f} minutes")

In [ ]:
# ============================================================
# CELL 7: Save Outputs (CRITICAL!)
# ============================================================

import os
import shutil
import json

print("=" * 60)
print("SAVING ALL OUTPUTS PERMANENTLY")
print("=" * 60)

src_dir = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire'
dst_dir = '/kaggle/working/final_models'
os.makedirs(dst_dir, exist_ok=True)

if os.path.exists(src_dir):
    # Copy all checkpoints with descriptive names
    for f in os.listdir(src_dir):
        if f.endswith('.pth'):
            src = os.path.join(src_dir, f)
            
            if f == 'best_ckpt.pth':
                dst_name = 'lcam_yolox_wildfire_v3_BEST.pth'
            elif f == 'latest_ckpt.pth':
                dst_name = 'lcam_yolox_wildfire_v3_LATEST.pth'
            else:
                dst_name = f'lcam_yolox_wildfire_v3_{f}'
            
            dst = os.path.join(dst_dir, dst_name)
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1e6
            print(f"  ✓ {dst_name}: {size_mb:.1f} MB")
    
    # Save LCAM source
    lcam_src = '/kaggle/working/YOLOX/yolox/models/lcam.py'
    if os.path.exists(lcam_src):
        shutil.copy2(lcam_src, os.path.join(dst_dir, 'lcam_v3.py'))
        print(f"  ✓ lcam_v3.py")
    
    # Save patched darknet
    darknet_src = '/kaggle/working/YOLOX/yolox/models/darknet.py'
    if os.path.exists(darknet_src):
        shutil.copy2(darknet_src, os.path.join(dst_dir, 'darknet_patched.py'))
        print(f"  ✓ darknet_patched.py")
    
    # Save config
    exp_src = '/kaggle/working/YOLOX/exps/custom/lcam_yolox_wildfire.py'
    if os.path.exists(exp_src):
        shutil.copy2(exp_src, os.path.join(dst_dir, 'lcam_yolox_wildfire_config.py'))
        print(f"  ✓ config.py")
    
    # Save training log
    log_src = os.path.join(src_dir, 'train_log.txt')
    if os.path.exists(log_src):
        shutil.copy2(log_src, os.path.join(dst_dir, 'train_log.txt'))
        print(f"  ✓ train_log.txt")
    
    # Metadata
    metadata = {
        "experiment": "LCAM-YOLOX v3.1 DPU-Optimized",
        "epochs_trained": 30,
        "final_mAP_50": 0.7650,
        "final_mAP_50_95": 0.4300,
        "smoke_AP": 0.485,
        "fire_AP": 0.374,
        "inference_time_ms": 8.80,
        "model_params": "8.98M",
        "model_gflops": 26.26,
        "classes": ["smoke", "fire"],
        "dataset": "D-Fire",
        "training_time_minutes": 136,
        "notes": "DPU-compatible LCAM, no torch.amax in code"
    }
    
    with open(os.path.join(dst_dir, 'metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"  ✓ metadata.json")

print(f"\n{'='*60}")
print("FILES IN /kaggle/working/final_models/")
print(f"{'='*60}")
for f in sorted(os.listdir(dst_dir)):
    path = os.path.join(dst_dir, f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

print(f"\n{'='*60}")
print("⚠⚠⚠ NEXT STEP: CLICK SAVE VERSION! ⚠⚠⚠")
print(f"{'='*60}")
print("1. Look at TOP RIGHT corner of Kaggle notebook")
print("2. Find 'Save Version' button")
print("3. CLICK IT NOW")
print("4. Choose 'Quick Save'")
print("5. Wait for confirmation")
print("\nWithout 'Save Version', files will be DELETED!")

In [ ]:
# Run this in a new cell to verify:
import os

# Check the file exists and has content
path = '/kaggle/working/final_models/lcam_v3.py'
if os.path.exists(path):
    size = os.path.getsize(path)
    print(f"File exists: {path}")
    print(f"Size: {size} bytes ({size/1024:.1f} KB)")
    
    # Show first 30 lines
    with open(path) as f:
        for i, line in enumerate(f.readlines()[:30], 1):
            print(f"{i:3d}: {line.rstrip()}")
else:
    print(f"NOT FOUND: {path}")
    print("\nFiles in final_models/:")
    for f in sorted(os.listdir('/kaggle/working/final_models')):
        print(f"  {f}")

In [ ]:
# ============================================================
# CELL 7.5: Patch demo.py for newer PyTorch
# ============================================================

demo_path = '/kaggle/working/YOLOX/tools/demo.py'

with open(demo_path, 'r') as f:
    content = f.read()

# Replace torch.load to use weights_only=False
old = 'ckpt = torch.load(ckpt_file, map_location="cpu")'
new = 'ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)'

if old in content:
    content = content.replace(old, new)
    with open(demo_path, 'w') as f:
        f.write(content)
    print("✓ demo.py patched")
else:
    # Check if already patched
    if new in content:
        print("✓ demo.py already patched")
    else:
        print("⚠ Pattern not found, showing torch.load lines:")
        for i, line in enumerate(content.split('\n'), 1):
            if 'torch.load' in line:
                print(f"  Line {i}: {line.strip()}")

In [ ]:
# ============================================================
# CELL 8: Test Trained Model on Real Image
# ============================================================

import os
import sys
import subprocess
from IPython.display import Image, display

print("=" * 60)
print("TESTING TRAINED MODEL ON REAL FIRE IMAGE")
print("=" * 60)

YOLOX_DIR = '/kaggle/working/YOLOX'
BEST_CKPT = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire/best_ckpt.pth'
EXP_FILE = '/kaggle/working/YOLOX/exps/custom/lcam_yolox_wildfire.py'
TEST_IMAGE = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/AoF06723.jpg'

print(f"\nFiles to use:")
print(f"  Model:  {os.path.exists(BEST_CKPT)} - {BEST_CKPT}")
print(f"  Config: {os.path.exists(EXP_FILE)} - {EXP_FILE}")
print(f"  Image:  {os.path.exists(TEST_IMAGE)} - {TEST_IMAGE}")

# Use YOLOX's built-in demo tool
os.chdir(YOLOX_DIR)
env = os.environ.copy()
env['PYTHONPATH'] = f'{YOLOX_DIR}:{env.get("PYTHONPATH", "")}'

cmd = [
    'python3', 'tools/demo.py', 'image',
    '-f', EXP_FILE,
    '-c', BEST_CKPT,
    '--path', TEST_IMAGE,
    '--conf', '0.25',          # Confidence threshold
    '--nms', '0.45',           # NMS threshold  
    '--tsize', '640',          # Input size
    '--save_result',           # Save output image
    '--device', 'gpu',
]

print(f"\n{'='*60}")
print("Running detection...")
print(f"{'='*60}\n")

result = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-2000:])  # Last 2000 chars

# Find the output image
output_dir = '/kaggle/working/YOLOX/YOLOX_outputs/lcam_yolox_wildfire/vis_res'
if os.path.exists(output_dir):
    # Find most recent folder
    folders = sorted(os.listdir(output_dir))
    if folders:
        latest = os.path.join(output_dir, folders[-1])
        print(f"\n{'='*60}")
        print(f"Latest output folder: {latest}")
        print(f"{'='*60}")
        
        # Find detection image
        for f in os.listdir(latest):
            if f.endswith('.jpg'):
                detection_path = os.path.join(latest, f)
                print(f"\nDetection result: {detection_path}")
                
                # Display the image inline
                display(Image(detection_path))
                
                # Also copy to easy-to-find location
                import shutil
                target = f'/kaggle/working/detection_{f}'
                shutil.copy2(detection_path, target)
                print(f"Copied to: {target}")
                break
else:
    print("\n⚠ Output folder not found. Check stderr above.")

In [ ]:
# ============================================================
# CELL 9: Display Detection Result
# ============================================================

import os
import shutil
from IPython.display import Image, display

# The correct output location
base_output = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire/vis_res'

if os.path.exists(base_output):
    # Find the most recent folder
    folders = sorted([f for f in os.listdir(base_output) 
                      if os.path.isdir(os.path.join(base_output, f))])
    
    print(f"Found {len(folders)} detection folder(s)")
    
    if folders:
        latest = os.path.join(base_output, folders[-1])
        print(f"Latest folder: {latest}")
        
        # List files
        files = os.listdir(latest)
        print(f"Files: {files}")
        
        # Display each image
        for f in files:
            if f.endswith(('.jpg', '.png', '.jpeg')):
                img_path = os.path.join(latest, f)
                size_kb = os.path.getsize(img_path) / 1024
                
                print(f"\n{'='*60}")
                print(f"Detection result: {f}")
                print(f"Size: {size_kb:.1f} KB")
                print(f"{'='*60}\n")
                
                # Display inline
                display(Image(img_path))
                
                # Copy to easy location for download
                target = f'/kaggle/working/detection_result.jpg'
                shutil.copy2(img_path, target)
                print(f"\n✓ Copied to: {target}")
                print(f"  Download from Output panel to see locally")
else:
    print(f"⚠ Directory not found: {base_output}")

In [ ]:
# ============================================================
# CELL 10: Check Labels + Try Multiple Images
# ============================================================

import os

DFIRE_PATH = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data'

# Check label for AoF06723
label_file = f'{DFIRE_PATH}/test/labels/AoF06723.txt'
print(f"Checking labels for AoF06723.jpg:")
if os.path.exists(label_file):
    with open(label_file) as f:
        content = f.read().strip()
    if content:
        print(f"  Has labels:")
        for line in content.split('\n'):
            parts = line.split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                cls_name = "smoke" if cls_id == 0 else "fire"
                print(f"    Class: {cls_name} ({cls_id})")
                print(f"    Bbox: cx={parts[1]}, cy={parts[2]}, w={parts[3]}, h={parts[4]}")
    else:
        print(f"  EMPTY label file - NO ground truth detections!")
        print(f"  → This image has NO fire/smoke (negative sample)")
else:
    print(f"  No label file - NO ground truth detections!")

# Find images that DO have detections
print(f"\n{'='*60}")
print("Finding test images with smoke/fire labels...")
print(f"{'='*60}")

test_labels = f'{DFIRE_PATH}/test/labels'
images_with_detections = []

for label_file in sorted(os.listdir(test_labels))[:200]:
    if not label_file.endswith('.txt'):
        continue
    
    path = os.path.join(test_labels, label_file)
    with open(path) as f:
        content = f.read().strip()
    
    if content:
        # Has at least one detection
        classes = set()
        for line in content.split('\n'):
            parts = line.split()
            if len(parts) >= 5:
                classes.add(int(parts[0]))
        
        if classes:
            images_with_detections.append({
                'name': label_file.replace('.txt', '.jpg'),
                'classes': classes,
                'num_objects': len(content.split('\n'))
            })

print(f"\nFound {len(images_with_detections)} images with detections (first 200 checked)")
print(f"\nFirst 10 with fire/smoke:")
for item in images_with_detections[:10]:
    class_names = []
    if 0 in item['classes']: class_names.append('smoke')
    if 1 in item['classes']: class_names.append('fire')
    print(f"  {item['name']}: {item['num_objects']} {'/'.join(class_names)}")

In [ ]:
# ============================================================
# CELL 11: Deeper Dataset Investigation
# ============================================================

import os

DFIRE_PATH = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data'

# Check structure
print("Dataset structure:")
print(f"  Path: {DFIRE_PATH}")
print(f"  Exists: {os.path.exists(DFIRE_PATH)}")

for split in ['train', 'val', 'test']:
    img_dir = f'{DFIRE_PATH}/{split}/images'
    lbl_dir = f'{DFIRE_PATH}/{split}/labels'
    
    n_imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    
    # Count non-empty labels
    n_with_objects = 0
    if os.path.exists(lbl_dir):
        for f in os.listdir(lbl_dir):
            if f.endswith('.txt'):
                path = os.path.join(lbl_dir, f)
                if os.path.getsize(path) > 0:
                    n_with_objects += 1
    
    print(f"\n{split}:")
    print(f"  Images: {n_imgs}")
    print(f"  Label files: {n_lbls}")
    print(f"  Labels WITH objects: {n_with_objects}")
    print(f"  Labels EMPTY (no objects): {n_lbls - n_with_objects}")

# Look at a few label files - show their content
print(f"\n{'='*60}")
print("Sample of TEST labels content:")
print(f"{'='*60}")

lbl_dir = f'{DFIRE_PATH}/test/labels'
shown_with_obj = 0
shown_empty = 0

for f in sorted(os.listdir(lbl_dir)):
    path = os.path.join(lbl_dir, f)
    size = os.path.getsize(path)
    
    if size > 0 and shown_with_obj < 3:
        print(f"\n  {f} ({size} bytes):")
        with open(path) as fp:
            content = fp.read().strip()
        for line in content.split('\n')[:3]:
            print(f"    {line}")
        shown_with_obj += 1
    elif size == 0 and shown_empty < 2:
        print(f"\n  {f}: EMPTY")
        shown_empty += 1
    
    if shown_with_obj >= 3 and shown_empty >= 2:
        break

# Find images that have labels with objects
print(f"\n{'='*60}")
print("First 10 test images WITH fire/smoke labels:")
print(f"{'='*60}")

found = 0
for f in sorted(os.listdir(lbl_dir)):
    if f.endswith('.txt'):
        path = os.path.join(lbl_dir, f)
        if os.path.getsize(path) > 0:
            with open(path) as fp:
                content = fp.read().strip()
            
            classes = set()
            for line in content.split('\n'):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        classes.add(int(parts[0]))
                    except ValueError:
                        pass
            
            if classes:
                class_names = []
                if 0 in classes: class_names.append('smoke')
                if 1 in classes: class_names.append('fire')
                
                img_name = f.replace('.txt', '.jpg')
                img_path = f'{DFIRE_PATH}/test/images/{img_name}'
                exists = "✓" if os.path.exists(img_path) else "✗"
                
                print(f"  {img_name} {exists}: {'+'.join(class_names)}")
                found += 1
                if found >= 10:
                    break

if found == 0:
    print("  ⚠ STILL no images with labels found - check label format")

In [ ]:
# ============================================================
# CELL 12: Detection on Real Fire+Smoke Image
# ============================================================

import os
import shutil
import subprocess
from IPython.display import Image, display

print("=" * 60)
print("DETECTION ON IMAGE WITH ACTUAL FIRE + SMOKE")
print("=" * 60)

YOLOX_DIR = '/kaggle/working/YOLOX'
BEST_CKPT = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire/best_ckpt.pth'
EXP_FILE = '/kaggle/working/YOLOX/exps/custom/lcam_yolox_wildfire.py'
TEST_IMAGE = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/AoF07718.jpg'

print(f"\nTesting on: AoF07718.jpg")
print(f"Expected: BOTH smoke AND fire")

os.chdir(YOLOX_DIR)
env = os.environ.copy()
env['PYTHONPATH'] = f'{YOLOX_DIR}:{env.get("PYTHONPATH", "")}'

cmd = [
    'python3', 'tools/demo.py', 'image',
    '-f', EXP_FILE,
    '-c', BEST_CKPT,
    '--path', TEST_IMAGE,
    '--conf', '0.25',
    '--nms', '0.45',
    '--tsize', '640',
    '--save_result',
    '--device', 'gpu',
]

print(f"\nRunning detection...\n")
result = subprocess.run(cmd, env=env, capture_output=True, text=True)

# Print key lines from stderr (logs)
for line in result.stderr.split('\n'):
    if 'Infer time' in line or 'Saving' in line or 'ERROR' in line or 'Traceback' in line:
        print(line)

# Find and display result
base_output = '/kaggle/working/YOLOX/YOLOX_outputs/lcam_yolox_wildfire/vis_res'
if os.path.exists(base_output):
    folders = sorted([f for f in os.listdir(base_output) 
                      if os.path.isdir(os.path.join(base_output, f))])
    
    if folders:
        latest = os.path.join(base_output, folders[-1])
        for f in os.listdir(latest):
            if f == 'AoF07718.jpg':
                img_path = os.path.join(latest, f)
                print(f"\n{'='*60}")
                print(f"DETECTION RESULT - LOOK BELOW")
                print(f"{'='*60}\n")
                
                display(Image(img_path))
                
                # Copy for download
                target = '/kaggle/working/detection_fire_smoke.jpg'
                shutil.copy2(img_path, target)
                print(f"\n✓ Saved to: {target}")
                break

In [ ]:
# ============================================================
# CELL 13: Display Detection Result (Fixed Path)
# ============================================================

import os
import shutil
from IPython.display import Image, display

# CORRECT path (from the log message above)
base_output = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire/vis_res'

print(f"Looking in: {base_output}")
print(f"Exists: {os.path.exists(base_output)}\n")

if os.path.exists(base_output):
    folders = sorted([f for f in os.listdir(base_output) 
                      if os.path.isdir(os.path.join(base_output, f))])
    
    print(f"Found {len(folders)} detection folders\n")
    
    # Get the most recent (last) folder
    if folders:
        latest = os.path.join(base_output, folders[-1])
        print(f"Latest folder: {folders[-1]}")
        print(f"Files in folder:")
        
        for f in os.listdir(latest):
            img_path = os.path.join(latest, f)
            size_kb = os.path.getsize(img_path) / 1024
            print(f"  {f}: {size_kb:.1f} KB")
            
            if f.endswith(('.jpg', '.png', '.jpeg')):
                print(f"\n{'='*60}")
                print(f"DISPLAYING: {f}")
                print(f"{'='*60}\n")
                
                display(Image(img_path))
                
                # Save easy-access copy
                target = '/kaggle/working/detection_fire_smoke.jpg'
                shutil.copy2(img_path, target)
                print(f"\n✓ Copied to: {target}")
else:
    print("Directory not found!")

In [ ]:
# ============================================================
# CELL 14: Fix Class Labels in YOLOX Demo
# ============================================================

import os

# YOLOX uses COCO classes by default
# We need to override with our 2 classes: smoke, fire

# Check the demo.py for class names
demo_path = '/kaggle/working/YOLOX/tools/demo.py'

with open(demo_path, 'r') as f:
    content = f.read()

# Find and show COCO_CLASSES usage
print("Searching for COCO_CLASSES in demo.py...")
if 'COCO_CLASSES' in content:
    # Show relevant lines
    for i, line in enumerate(content.split('\n'), 1):
        if 'COCO_CLASSES' in line:
            print(f"  Line {i}: {line.strip()}")

# Replace COCO_CLASSES import with our custom classes
old_import = "from yolox.data.datasets import COCO_CLASSES"
new_import = '''from yolox.data.datasets import COCO_CLASSES
# Override with custom classes for wildfire detection
COCO_CLASSES = ("smoke", "fire")'''

if old_import in content and new_import not in content:
    content = content.replace(old_import, new_import)
    with open(demo_path, 'w') as f:
        f.write(content)
    print("\n✓ demo.py patched with custom class names")
else:
    if new_import in content:
        print("\n✓ demo.py already has custom classes")
    else:
        print("\n⚠ Could not find COCO_CLASSES import")
        print("Showing import section:")
        for i, line in enumerate(content.split('\n')[:30], 1):
            if 'import' in line:
                print(f"  {i}: {line}")

In [ ]:
# ============================================================
# CELL 15: Detection With Correct smoke/fire Labels
# ============================================================

import os
import shutil
import subprocess
from IPython.display import Image, display

print("=" * 60)
print("DETECTION WITH CORRECT LABELS (smoke, fire)")
print("=" * 60)

YOLOX_DIR = '/kaggle/working/YOLOX'
BEST_CKPT = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire/best_ckpt.pth'
EXP_FILE = '/kaggle/working/YOLOX/exps/custom/lcam_yolox_wildfire.py'
TEST_IMAGE = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/AoF07718.jpg'

os.chdir(YOLOX_DIR)
env = os.environ.copy()
env['PYTHONPATH'] = f'{YOLOX_DIR}:{env.get("PYTHONPATH", "")}'

cmd = [
    'python3', 'tools/demo.py', 'image',
    '-f', EXP_FILE,
    '-c', BEST_CKPT,
    '--path', TEST_IMAGE,
    '--conf', '0.25',
    '--nms', '0.45',
    '--tsize', '640',
    '--save_result',
    '--device', 'gpu',
]

print("Running detection...\n")
result = subprocess.run(cmd, env=env, capture_output=True, text=True)

# Print key log lines
for line in result.stderr.split('\n'):
    if 'Infer time' in line or 'Saving' in line or 'ERROR' in line:
        print(line)

# Find and display latest result
base_output = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire/vis_res'
if os.path.exists(base_output):
    folders = sorted([f for f in os.listdir(base_output) 
                      if os.path.isdir(os.path.join(base_output, f))])
    
    if folders:
        latest = os.path.join(base_output, folders[-1])
        print(f"\nLatest result folder: {folders[-1]}")
        
        for f in os.listdir(latest):
            if f == 'AoF07718.jpg':
                img_path = os.path.join(latest, f)
                print(f"\n{'='*60}")
                print(f"FIXED DETECTION RESULT - LOOK BELOW")
                print(f"{'='*60}\n")
                
                display(Image(img_path))
                
                # Copy for download
                target = '/kaggle/working/detection_fixed_labels.jpg'
                shutil.copy2(img_path, target)
                print(f"\n✓ Saved to: {target}")
                break